In [61]:
import os
import numpy as np
from skimage.io import imread, imsave

In [65]:
grid =2
dir = "segmentation"
output_mask_file = "z3_nulei_segmentation.tif"

In [18]:
def matfileName(gridR, gridC):
    return os.path.join(dir, str(gridR) + str(gridC) +"_cp_masks.tif")

In [35]:
def mask_index(gridR, gridC, gridSize):
    return gridR * gridSize + gridC

In [44]:
def fix_bet_Col_mask(mask_left, mask_right):
    h,w = mask_left.shape
    fixed_mask_right = mask_right[:]
    for i in range (0, h):
        leftV = mask_left[i, -1]
        rightV = mask_right[i, 0]
        if leftV ==0 or rightV == 0 or leftV == rightV:
            pass
        else:
            fixed_mask_right[fixed_mask_right == rightV] = leftV
    return fixed_mask_right

In [46]:
def fix_bet_Row_mask(mask_top, mask_bottom):
    h,w = mask_top.shape
    fixed_mask_bottom = mask_bottom[:]
    for i in range (0, w):
        topV = mask_top[h-1, i]
        bottomV = mask_bottom[0, i]
        if topV ==0 or bottomV == 0 or topV == bottomV:
            pass
        else:
            fixed_mask_bottom[fixed_mask_bottom == bottomV] = topV
    return fixed_mask_bottom

### get cell number per grid and mask offsets

In [30]:
cellNumGrid = np.zeros((grid, grid), dtype =int)
cellNumOffset = np.zeros((grid, grid), dtype =int)
cellNumGrid, cellNumOffset

(array([[0, 0],
        [0, 0]]),
 array([[0, 0],
        [0, 0]]))

In [31]:
totalCell = 0 
for gridR in range (0, grid):
    for gridC in range (0, grid):
        cellNumOffset[gridR, gridC] = totalCell
        matFile = matfileName(gridR, gridC)
        mask = imread(matFile)
        cellN = np.max(mask) 
        cellNumGrid[gridR, gridC] = cellN
        totalCell = totalCell + cellN
cellNumGrid, cellNumOffset

(array([[9374, 7921],
        [7591, 9609]]),
 array([[    0,  9374],
        [17295, 24886]]))

### generate grid mask with offset

In [41]:
masks = [None] * grid * grid
for gridR in range (0, grid):
    for gridC in range (0, grid):
        offset = cellNumOffset[gridR, gridC]
        matFile = matfileName(gridR, gridC)
        mask = imread(matFile)
        mask[mask != 0] += offset
        index = mask_index(gridR, gridC, grid)
        masks[index] = mask
        print(np.max(mask))
masks

9374
17295
24886
34495


[array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=uint16),
 array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=uint16),
 array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=uint16),
 array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=uint16)]

### fix the border between grids, for each row, between columns

In [45]:
for gridR in range (0, grid):
    for gridC in range (0, grid-1):
        print(gridR, gridC)
        index_1 = mask_index(gridR, gridC, grid)
        mask_left = masks[index_1]
        index_2 = mask_index(gridR, gridC+1, grid)
        mask_right = masks[index_2]
        
        fixed_mask_right = fix_bet_Col_mask(mask_left, mask_right)
        masks[index_2] = fixed_mask_right


0 0
1 0


### fix the border between grids, for each col, between rows

In [47]:
for gridC in range (0, grid):
    for gridR in range (0, grid-1):
        print(gridR, gridC)
        index_1 = mask_index(gridR, gridC, grid)
        mask_top = masks[index_1]
        index_2 = mask_index(gridR+1, gridC, grid)
        mask_bottom = masks[index_2]
        
        fixed_mask_bottom = fix_bet_Row_mask(mask_top, mask_bottom)
        masks[index_2] = fixed_mask_bottom

0 0
0 1


### merge masks

In [49]:
gridC = 0
total_h = 0
for gridR in range (0, grid):
    index = mask_index(gridR, gridC, grid)
    mask = masks[index]
    h, w = mask.shape
    total_h += h
    
gridR = 0
total_w = 0
for gridC in range (0, grid):
    index = mask_index(gridR, gridC, grid)
    mask = masks[index]
    h, w = mask.shape
    total_w += w
    
total_h, total_w

(69762, 69741)

In [50]:
np.max(masks[-1])

34495

In [58]:
merged_mask = np.zeros((total_h, total_w), dtype = np.uint16)

start_h =0 
for gridR in range (0, grid):
    start_w = 0
    for gridC in range (0, grid):
        index = mask_index(gridR, gridC, grid)
        mask = masks[index]
        h,w = mask.shape
        print (h,w, start_h, start_h+h, start_w, start_w +w)
        merged_mask[start_h: start_h+h, start_w: start_w +w] = mask[:]
        start_w = start_w + w
    start_h = start_h + h

merged_mask

34881 34870 0 34881 0 34870
34881 34871 0 34881 34870 69741
34881 34870 34881 69762 0 34870
34881 34871 34881 69762 34870 69741


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint16)

### export merged_mask

In [64]:
imsave(output_mask_file, merged_mask )